# 06 Modeling

Treinamento e avaliação de modelos de risco de crédito com Logistic Regression e LightGBM.

## Abordagem

Este notebook treina e avalia modelos de classificação de risco usando o dataset preparado. Também adiciona validação cruzada para testar a robustez dos resultados.


In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path('..').resolve()))

import pandas as pd
from src.modeling import train_models

model_dataset = pd.read_parquet(Path('..') / 'data' / 'processed' / 'model_dataset.parquet')
results = train_models(model_dataset, target_col='target')
print('Training results:')
print(results['results'])

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from src.modeling import load_model
from src.feature_engineering import prepare_model_dataset

model = load_model('final_model.pkl')
feature_names = model.booster_.feature_name_ if hasattr(model, 'booster_') else None
importances = model.feature_importances_ if hasattr(model, 'feature_importances_') else None
if feature_names is not None and importances is not None:
    importance_df = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values('importance', ascending=False)
    display(importance_df.head(15))

# Validação cruzada para checagem de robustez
model_dataset = pd.read_parquet(Path('..') / 'data' / 'processed' / 'model_dataset.parquet')
X, y, _ = prepare_model_dataset(model_dataset, target_col='target')
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {'roc_auc': 'roc_auc', 'accuracy': 'accuracy'}

lr = LogisticRegression(max_iter=2000, solver='saga', random_state=42)
lgbm = LGBMClassifier(random_state=42, n_estimators=200)

lr_cv = cross_validate(lr, X, y, scoring=scoring, cv=skf, return_train_score=False)
lgbm_cv = cross_validate(lgbm, X, y, scoring=scoring, cv=skf, return_train_score=False)

print('Logistic CV ROC AUC:', np.mean(lr_cv['test_roc_auc']))
print('LightGBM CV ROC AUC:', np.mean(lgbm_cv['test_roc_auc']))
print('LightGBM CV accuracy:', np.mean(lgbm_cv['test_accuracy']))

cv_results = pd.DataFrame({
    'logistic_roc_auc': lr_cv['test_roc_auc'],
    'lgbm_roc_auc': lgbm_cv['test_roc_auc'],
    'lgbm_accuracy': lgbm_cv['test_accuracy'],
})

display(cv_results.describe())

## Interpretação inicial

Os principais drivers de risco devem incluir alavancagem de renda, histórico de atraso e tamanho da operação.
A combinação de um modelo linear e um modelo de árvore permite comparar robustez e performance preditiva.